# EuroCropML - Step 0: Upload Data to HuggingFace
One-time setup: download from Zenodo, push to your HuggingFace repo.
After this, notebooks 01+ download from HF instead of Zenodo.

In [ ]:
!pip install huggingface_hub -q

In [ ]:
# Login to HuggingFace (run once per session)
from huggingface_hub import login
login()  # will prompt for token

In [ ]:
# Download from Zenodo with aria2c (fast)
!apt-get install -y aria2 -q

import requests, os

record_id = 15095445
record = requests.get(f"https://zenodo.org/api/records/{record_id}").json()

for file in record["files"]:
    if file["key"] not in ["preprocess.zip", "split.zip"]:
        continue
    filename = file["key"]
    filesize = file["size"]
    url = file["links"]["self"]

    if os.path.exists(filename) and os.path.getsize(filename) == filesize:
        print(f"Already downloaded: {filename}")
    else:
        print(f"Downloading: {filename} ({filesize / 1e6:.1f} MB)")
        !aria2c -x 16 -s 16 -o "{filename}" "{url}"

print("Done!")
!ls -lh *.zip

In [ ]:
# Create HuggingFace repo and upload
from huggingface_hub import HfApi

REPO_ID = "mahdi555/eurocrop-data"  # change to your repo

api = HfApi()
try:
    api.create_repo(repo_id=REPO_ID, exist_ok=True)
    print(f"Repo ready: https://huggingface.co/{REPO_ID}")
except Exception as e:
    print(f"Repo creation: {e}")

for f in ["preprocess.zip", "split.zip"]:
    print(f"Uploading {f}...")
    api.upload_file(
        path_or_fileobj=f,
        path_in_repo=f,
        repo_id=REPO_ID,
        repo_type="model"
    )
    print(f"  Uploaded {f}")

print(f"\nAll done! Data is at: https://huggingface.co/{REPO_ID}")